In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score, roc_curve, precision_recall_curve
from imblearn.over_sampling import SMOTE
import shap
import os
import joblib
from printer import Printer
import json

/home/jxiao/anaconda3/envs/m2f/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load data
Label meaning: 0 = negative (normal), 1 = positive (patient)

In [ ]:
### load data
data_path = '/data/gas_new/values_refine/RGB_diff_refine_label_new.csv'
# data_path = '/data/gas_new/values_refine/RGB_diff_refine_label_new_n2.csv'
diff = pd.read_csv(data_path, index_col='id')
diff_n2 = (np.sqrt(diff.iloc[:, :-1]**2)).T.groupby(np.arange(diff.shape[1]-1) // 3).sum().T

X = diff.drop('label', axis=1, inplace=False).to_numpy()
X_n2 = diff_n2.to_numpy()
y = diff['label'].to_numpy().astype(int)
print(len(y), sum(y))

(134, np.int64(81))

## Training configuration and model initialization
$\bullet$ If a weight directory is specified, the training logs and model weights will be saved in overwrite mode.  
$\bullet$ By setting weight_dir to None or an empty string to disable logging and saving.

In [ ]:
INPUT_DIM = 8 if "n2" in data_path else 24
EPOCHS = 250
BATCH_SIZE = 20
LR = 2e-4
PATIENCE = 10  # early stopping
N_SPLITS = 5
THRESHOLD = 0.5
DROPOUT = 0.3
weight_dir = 'weights/RGB_diff_refine_label_new'
use_smote = False
use_posweight = False
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if weight_dir:
    if not os.path.exists(weight_dir):
        os.makedirs(weight_dir)
        
    logger = Printer(os.path.join(weight_dir, 'log.txt'))

    with open(os.path.join(weight_dir, 'config.txt'), 'w') as f:
        f.write(f"data_path:{data_path}\n")
        f.write(f"INPUT_DIM: {INPUT_DIM}\n")
        f.write(f"EPOCHS: {EPOCHS}\n")
        f.write(f"BATCH_SIZE: {BATCH_SIZE}\n")
        f.write(f"LR: {LR}\n")
        f.write(f"PATIENCE: {PATIENCE}\n")
        f.write(f"N_SPLITS: {N_SPLITS}\n")
        f.write(f"THRESHOLD: {THRESHOLD}\n")
        f.write(f"DROPOUT: {DROPOUT}\n")
        f.write(f"use_smote: {use_smote}\n")
        f.write(f"use_posweight: {use_posweight}\n")

class SmallMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),  # 64
            nn.ReLU(),
            nn.Dropout(DROPOUT),  # 0.3
            nn.Linear(64, 32),  # 32
            nn.ReLU(),
            nn.Dropout(DROPOUT),  # 0.3
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x)

## Training process

In [ ]:
skf = StratifiedKFold(n_splits=N_SPLITS, 
                      shuffle=True, 
                      random_state=42
                    )

all_acc, all_f1, all_auc, all_precision, all_recall = [], [], [], [], [] # eval metrics
all_fpr, all_tpr = [], []  # ROC curves
all_precision_thre, all_recall_thre, all_thresholds = [], [], []  # Precision / Recall vs. Threshold curves
y_val_list, y_pred_list, y_label_list, prob_list = [], [], [], []  # save ground truth and predictions
acc_per_epoch, f1_per_epoch, auc_per_epoch = [[] for _ in range(N_SPLITS)], [[] for _ in range(N_SPLITS)], [[] for _ in range(N_SPLITS)] # model performance evolution curves
train_loss_list, val_loss_list= [[] for _ in range(N_SPLITS)], [[] for _ in range(N_SPLITS)] # loss curves

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    if weight_dir:
        logger.write(f"\n===== Fold {fold + 1} =====")
    
    print(f"\n===== Fold {fold + 1} =====")

    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    # --- normalization ---
    scaler = StandardScaler()
    scaler.fit(X_train)

    X_train = scaler.transform(X_train)
    X_val = scaler.transform(X_val)

    # --- SMOTE ---
    if use_smote:
        smote = SMOTE(random_state=42)
        X_train, y_train = smote.fit_resample(X_train, y_train)

    X_train = torch.tensor(X_train, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
    X_val = torch.tensor(X_val, dtype=torch.float32)
    y_val = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)

    # --- DataLoader ---
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=BATCH_SIZE)

    # --- Model & Optimizer ---
    model = SmallMLP(INPUT_DIM).to(device)
    if use_posweight:
        num_pos = y_train.sum()
        num_neg = y_train.shape[0] - num_pos
        # pos_weight = torch.tensor([num_neg / num_pos]).to(device)  # 正例的权重
        pos_weight = torch.tensor([3.0]).to(device)  # 手动设定正例权重
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    else:
        criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    # optimizer = optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=1e-4)

    # --- Train ---
    best_val_loss = float('inf')
    patience_counter = 0

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * xb.size(0)
        train_loss /= len(train_loader.dataset)

        #--- Validation ---
        model.eval()
        val_loss = 0.0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = model(xb)
                loss = criterion(preds, yb)
                val_loss += loss.item() * xb.size(0)
                all_preds.append(torch.sigmoid(preds).cpu().numpy())
                all_labels.append(yb.cpu().numpy())
        val_loss /= len(val_loader.dataset)

        all_preds = np.vstack(all_preds)
        all_labels = np.vstack(all_labels)
        pred_labels = (all_preds > 0.5).astype(int)

        acc = accuracy_score(all_labels, pred_labels)
        f1 = f1_score(all_labels, pred_labels)
        auc = roc_auc_score(all_labels, all_preds)

        acc_per_epoch[fold].append(acc)
        f1_per_epoch[fold].append(f1)
        auc_per_epoch[fold].append(auc)

        train_loss_list[fold].append(train_loss)
        val_loss_list[fold].append(val_loss)
        
        if weight_dir:
            logger.write(f"Epoch [{epoch+1:03d}] "
                f"TrainLoss={train_loss:.4f} ValLoss={val_loss:.4f} "
                f"Acc={acc:.3f} F1={f1:.3f} AUC={auc:.3f}")

        #--- Early stopping ---
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_weights = model.state_dict()
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE or epoch == EPOCHS - 1:
                print("Stop in epoch {}!".format(epoch + 1))
                if weight_dir:
                    logger.write("Stopping in epoch {}!".format(epoch + 1))
                    weight_path = os.path.join(weight_dir, f'best_model_fold_{fold + 1}.pth')
                    torch.save(best_weights, weight_path)
                    joblib.dump(scaler, os.path.join(weight_dir, f"scaler_fold_{fold + 1}.pkl"))
                    print(f"Model weights and scaler saved to {weight_dir}.")
                break
    model.load_state_dict(best_weights)

    # --- Final evaluation ---
    model.eval()
    with torch.no_grad():
        preds = torch.sigmoid(model(X_val.to(device))).cpu().numpy()
        pred_labels = (preds > THRESHOLD).astype(int)
    prob_list.append(preds)

    acc = accuracy_score(y_val, pred_labels)
    f1 = f1_score(y_val, pred_labels)
    auc = roc_auc_score(y_val, preds)
    precision = precision_score(y_val, pred_labels)
    recall = recall_score(y_val, pred_labels)
    
    fpr, tpr, _ = roc_curve(y_val, preds)
    precision_thre, recall_thre, thresholds = precision_recall_curve(y_val, preds)
    if weight_dir:
        with open(os.path.join(weight_dir, f'split_fold_{fold+1}.json'), 'w') as f:
            json.dump({
                'train_idx': train_idx.tolist(),
                'val_idx': val_idx.tolist()
            }, f)
        logger.write(f"Fold {fold + 1} val results: ACC={acc:.3f}, F1={f1:.3f}, precision={precision:.3f}, recall={recall:.3f}")
    print(f"Fold {fold + 1} val results: ACC={acc:.3f}, F1={f1:.3f}, precision={precision:.3f}, recall={recall:.3f}")
    all_acc.append(acc)
    all_f1.append(f1)
    all_auc.append(auc)
    all_precision.append(precision)
    all_recall.append(recall)

    all_fpr.append(fpr)
    all_tpr.append(tpr)
    all_precision_thre.append(precision_thre)
    all_recall_thre.append(recall_thre)
    all_thresholds.append(thresholds)

    y_val_list.append(y_val.squeeze().cpu().tolist())
    y_pred_list.append(preds.squeeze().tolist())
    y_label_list.append(pred_labels.squeeze().tolist())


if weight_dir:
    logger.write("\n===== Overall Results =====")
    logger.write(f"Average ACC: {np.mean(all_acc):.4f} ± {np.std(all_acc):.4f}")
    logger.write(f"Average F1: {np.mean(all_f1):.4f} ± {np.std(all_f1):.4f}")
    logger.write(f"Average AUC: {np.mean(all_auc):.4f} ± {np.std(all_auc):.4f}")
    logger.write(f"Average Precision: {np.mean(all_precision):.4f} ± {np.std(all_precision):.4f}")
    logger.write(f"Average Recall: {np.mean(all_recall):.4f} ± {np.std(all_recall):.4f}")

In [ ]:
print(f"\n===== Overall Results ({N_SPLITS}-Fold Avg) =====")
print(f"ACC={np.mean(all_acc):.3f} ± {np.std(all_acc):.3f}")
print(f"F1 ={np.mean(all_f1):.3f} ± {np.std(all_f1):.3f}")
print(f"AUC={np.mean(all_auc):.3f} ± {np.std(all_auc):.3f}")
print(f"Precision={np.mean(all_precision):.3f} ± {np.std(all_precision):.3f}")
print(f"Recall={np.mean(all_recall):.3f} ± {np.std(all_recall):.3f}")

## Tracking and analysis

### Loss curve

In [ ]:
import seaborn as sns
sns.set_theme(style="white")

records = []
for i in range(len(train_loss_list)):
    fold_name = f"Fold {i+1}"
    for epoch, loss in enumerate(train_loss_list[i], start=1):
        records.append({"fold": fold_name, "epoch": epoch, "loss": loss, "set": "train"})
    for epoch, loss in enumerate(val_loss_list[i], start=1):
        records.append({"fold": fold_name, "epoch": epoch, "loss": loss, "set": "val"})

df = pd.DataFrame.from_records(records)

fig, axes = plt.subplots(1, len(train_loss_list), figsize=(20, 4), sharey=True)
for i, ax in enumerate(axes):
    fold_name = f"Fold {i+1}"
    sns.lineplot(data=df[df["fold"] == fold_name], x="epoch", y="loss", hue="set",
                ax=ax, palette=["#1f77b4", "#ff7f0e"])
    ax.set_title(fold_name, fontsize=14)
    ax.set_xlabel("Epoch", fontsize=14)
    if i == 0:
        ax.set_ylabel("Loss", fontsize=14)
    else:
        ax.set_ylabel("")
    ax.legend(loc="lower left")

fig.suptitle("Loss Curve", fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0.005, -0.05, 1, 0.999])
plt.show()

### Metric-Epoch curve

In [ ]:
import seaborn as sns
sns.set_theme(style="white")

records = []
for i in range(len(acc_per_epoch)):
    fold_name = f"Fold {i+1}"
    for epoch, acc in enumerate(acc_per_epoch[i], start=1):
        records.append({"fold": fold_name, "epoch": epoch, "metric": acc, "metric_name": "Accuracy"})
    for epoch, f1 in enumerate(f1_per_epoch[i], start=1):
        records.append({"fold": fold_name, "epoch": epoch, "metric": f1, "metric_name": "F1-Score"})
    for epoch, auc in enumerate(auc_per_epoch[i], start=1):
        records.append({"fold": fold_name, "epoch": epoch, "metric": auc, "metric_name": "AUC"})

df = pd.DataFrame.from_records(records)

fig, axes = plt.subplots(1, len(train_loss_list), figsize=(20, 4.2), sharey=True)
for i, ax in enumerate(axes):
    fold_name = f"Fold {i+1}"
    sns.lineplot(data=df[df["fold"] == fold_name], x="epoch", y="metric", hue="metric_name",
                ax=ax, palette=["#1f77b4", "#ff7f0e", "#2ca02c"])
    ax.set_title(fold_name, fontsize=14)
    ax.set_xlabel("Epoch", fontsize=14)
    # ax[i].grid(True, linestyle='--', alpha=0.5)
    if i == 0:
        ax.set_ylabel("Metric", fontsize=14)
    else:
        ax.set_ylabel("")
    ax.legend(loc="lower right")

fig.suptitle("Model Performance Evolution", fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0.005, -0.05, 1, 0.9999])
plt.show()

### ROC curve

In [ ]:
fig, ax = plt.subplots(1, 5, figsize=(20, 4), sharey=True)

for i in range(5):
    ax[i].plot(all_fpr[i], all_tpr[i], color='darkorange', lw=2, label=f'ROC curve (AUC = {all_auc[i]:.3f})')
    ax[i].plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--')
    ax[i].set_title(f'Fold {i+1}', fontsize=14)
    ax[i].set_xlabel('False Positive Rate', fontsize=14)
    ax[i].legend(loc='lower right')
    # ax[i].grid(True, linestyle='--', alpha=0.5)

# plt.legend(loc='lower right')
fig.suptitle('ROC Curve', fontsize=14, fontweight='bold')
fig.supylabel('True positive rate', fontsize=14)
plt.tight_layout(rect=[0.005, -0.05, 1, 0.999])
plt.show()

### Precision-Recall curve

In [ ]:
fig, ax = plt.subplots(1, 5, figsize=(20, 4), sharey=True)

## for convenience，the last point of thresholds is dropped. Then the length will be equal to that of precision/recall.
for i in range(5):
    ax[i].plot(all_thresholds[i][:-1], all_precision_thre[i][:-2], label='Precision')
    ax[i].plot(all_thresholds[i][:-1], all_recall_thre[i][:-2], label='Recall')
    ax[i].set_xlabel('Threshold', fontsize=14)
    ax[i].legend(loc='lower left')
    # ax[i].grid(True, linestyle='--', alpha=0.5)
    ax[i].set_xticks(np.arange(0, 1.1, 0.2))
fig.suptitle('Precision / Recall vs. Threshold', fontsize=14, fontweight='bold')
fig.supylabel('Score', fontsize=14)
plt.tight_layout(rect=[0.005, -0.05, 1, 0.999])
plt.show()

## SHAP analysis

In [ ]:
# --- prepare model ---
model = SmallMLP(INPUT_DIM).to(device)
device = next(model.parameters()).device
shap_value_list = []
X_val_list = []

root = 'weights/RGB_diff_refine_label_new_n2'
# --- prepare data ---
for fold in range(5):
    model_weights = torch.load(os.path.join(root, f'best_model_fold_{fold+1}.pth'), weights_only=True)
    model.load_state_dict(model_weights)
    model.eval()

    with open(os.path.join(root, f'split_fold_{fold+1}.json'), 'r') as f:
        js = json.load(f)
    X_train, X_val = X_n2[js['train_idx']], X_n2[js['val_idx']]

    X_train = torch.tensor(X_train, dtype=torch.float32, device=device)
    X_val = torch.tensor(X_val, dtype=torch.float32, device=device)

    # --- Use train set as background samples ---
    background = torch.tensor(shap.kmeans(X_train.cpu().numpy(), 10).data, dtype=torch.float32, device=device)

    # --- Build DeepExplainer ---
    explainer = shap.DeepExplainer(model, background)

    # --- calculate SHAP values ---
    shap_values = explainer.shap_values(X_val)

    # --- convert to numpy ---
    shap_values = shap_values[0] if isinstance(shap_values, list) else shap_values.squeeze()
    print(shap_values.shape)
    shap_value_list.append(shap_values)
    X_val_list.append(X_val.cpu().numpy())

shap_concat = np.vstack(shap_value_list)
X_concat = np.vstack(X_val_list)

np.save('weights/RGB_diff_refine_label_new_n2/shap_values.npy', shap_concat)
np.save('weights/RGB_diff_refine_label_new_n2/shap_X_val.npy', X_concat)

In [ ]:
# --- Plot summary for each fold and whole set ---
for fold in range(5):
    plt.figure()
    shap.summary_plot(shap_value_list[fold], features=X_val_list[fold], feature_names=diff.columns[:-1].tolist(), show=False)
    plt.savefig(f'weights/RGB_diff_refine_label_new_n2/shap_summary_plot_fold_{fold+1}.svg', bbox_inches="tight")

plt.figure()
shap.summary_plot(shap_concat, features=X_concat, feature_names=diff.columns[:-1].tolist(), show=False)
plt.savefig(f'weights/RGB_diff_refine_label_new_n2/shap_summary_plot.svg', bbox_inches="tight")

In [ ]:
shap_concat = np.load('shap_values.npy')
X_concat = np.load('shap_X_val.npy')
feature_names = [f'Dye{i}' for i in range(1, 9)] 
shap.summary_plot(shap_concat, features=X_concat, feature_names=feature_names, show=False)

plt.xlabel("SHAP value", fontsize=12)
plt.ylabel("Feature value", fontsize=14) 
plt.xticks(fontsize=12)
plt.yticks(fontsize=12) 
plt.title("SHAP Summary Plot", fontsize=16, fontweight='bold')
plt.savefig(f'weights/RGB_diff_refine_label_new_n2/shap_summary_plot_customized.svg', bbox_inches="tight")

In [ ]:
# --- Get feature importance ---
importance = np.abs(shap_concat).mean(axis=0)
features = np.arange(8)

sorted_idx = np.argsort(importance)[::-1]
importance = importance[sorted_idx]
features = features[sorted_idx]

feature_importance = {i: imp for i, imp in zip(features, importance)}
with open('weights/RGB_diff_refine_label_new_n2/feature_importance.json', 'w') as f:
    json.dump(feature_importance, f)

## Save results

In [11]:
import json
path = 'weights/RGB_diff_refine_label_new'

In [13]:
js_train = {}
js_train['train_loss_by_epoch'] = train_loss_list
js_train['val_loss_by_epoch'] = val_loss_list
js_train['accuracy_by_epoch'] = acc_per_epoch
js_train['f1_score_by_epoch'] = f1_per_epoch
js_train['mean_auc_by_epoch'] = auc_per_epoch

with open(os.path.join(path, 'train_log.json'), 'w') as f:
    json.dump(js_train, f)

js_val = {}
js_val['threshold'] = THRESHOLD
js_val['accuracy'] = all_acc
js_val['f1_score'] = all_f1
js_val['auc'] = all_auc
js_val['precision'] = all_precision
js_val['recall'] = all_recall
js_val['fpr'] = [fpr.tolist() for fpr in all_fpr]
js_val['tpr'] = [tpr.tolist() for tpr in all_tpr]
js_val['precision_thre'] = [prec.tolist() for prec in all_precision_thre]
js_val['recall_thre'] = [rec.tolist() for rec in all_recall_thre]
js_val['thresholds'] = [thre.tolist() for thre in all_thresholds]

with open(os.path.join(path, 'val_log.json'), 'w') as f:
    json.dump(js_val, f)

In [ ]:
feature_importance = {i: imp for i, imp in zip(features, importance)}

with open(os.path.join(path, 'feature_importance.json'), 'w') as f:
    json.dump(feature_importance, f)

In [ ]:
val_num = {}
val_num['y_label'] = y_val_list
val_num['y_prob'] = y_pred_list
val_num['y_pred'] = y_label_list
val_num['threshold'] = THRESHOLD

with open(os.path.join(path, 'val_num.json'), 'w') as f:
    json.dump(val_num, f)